# 🏀 Predicto Demo — March Machine Learning Mania 2026

**Demonstração completa do pipeline de predição probabilística com calibração de probabilidades.**

---

## 1️⃣ Setup e Importações

In [ ]:
import sys
import os

# Garante que o diretório raiz do projeto está no path
PROJECT_ROOT = os.path.abspath(os.path.join(os.path.dirname('__file__'), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import brier_score_loss, roc_auc_score, log_loss
from sklearn.calibration import calibration_curve
from sklearn.linear_model import LogisticRegression

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print('✅ Imports OK')

## 2️⃣ Configuração de Ambiente (Kaggle vs Local)

In [ ]:
from src.config import CONFIG

# Detecta ambiente e permite override manual
_KAGGLE_DIR = '/kaggle/input/march-machine-learning-mania-2026'
_LOCAL_CANDIDATES = [
    CONFIG.get('data_dir', ''),          # valor do config.py
    os.path.join(PROJECT_ROOT, 'data'),  # Predicto/data/
    os.path.expanduser('~/data/march-machine-learning-mania-2026'),
]

def _find_data_dir():
    if os.path.isdir(_KAGGLE_DIR):
        return _KAGGLE_DIR
    for p in _LOCAL_CANDIDATES:
        if p and os.path.isdir(p):
            return p
    return None

DATA_DIR = _find_data_dir()

if DATA_DIR is None:
    # ── EDITE AQUI se os dados estiverem em outro local ──────────────────────
    DATA_DIR = '/caminho/para/seus/dados'
    print(f'⚠️  Diretório de dados não encontrado automaticamente.')
    print(f'   Edite a variável DATA_DIR nesta célula com o caminho correto.')
else:
    print(f'✅ DATA_DIR = {DATA_DIR}')

# Override do config com o data_dir resolvido
CONFIG['data_dir'] = DATA_DIR

TARGET_SEASON    = CONFIG['target_season']
BACKTEST_SEASONS = CONFIG['backtest_seasons']

print(f'   target_season    = {TARGET_SEASON}')
print(f'   backtest_seasons = {BACKTEST_SEASONS}')

# Lista os arquivos disponíveis (confirma que o path está correto)
if os.path.isdir(DATA_DIR):
    print('\nArquivos de dados disponíveis:')
    for f in sorted(os.listdir(DATA_DIR)):
        print(f'  {f}')
else:
    print(f'\n❌ Diretório não existe: {DATA_DIR}')
    print('   Corrija DATA_DIR antes de continuar.')

## 3️⃣ Carregamento de Dados

In [ ]:
from src.data import load_regular_season_detailed, load_tourney_detailed, load_seeds

print('Carregando dados de jogos (regular season)...')
games_m = load_regular_season_detailed(DATA_DIR, 'M')
games_w = load_regular_season_detailed(DATA_DIR, 'W')

print(f'  Masculino: {games_m.shape}  |  Feminino: {games_w.shape}')
print(f'  Temporadas M: {sorted(games_m["Season"].unique())}')
print()

# Seeds
print('Carregando seeds do torneio...')
seeds_m = load_seeds(DATA_DIR, 'M')
seeds_w = load_seeds(DATA_DIR, 'W')
print(f'  Seeds M: {seeds_m.shape}  |  Seeds W: {seeds_w.shape}')

# Tourney
print('Carregando resultados do torneio...')
tourney_m = load_tourney_detailed(DATA_DIR, 'M')
tourney_w = load_tourney_detailed(DATA_DIR, 'W')
print(f'  Tourney M: {tourney_m.shape}  |  Tourney W: {tourney_w.shape}')

print('\n✅ Dados carregados com sucesso!')
games_m.head()

## 4️⃣ ELO Ratings — Cross-Seasonal com Carry-Forward

In [ ]:
from src.ratings import precompute_starting_elo

# Prepara estrutura de jogos para o Elo (vitórias com margem)
def _elo_games(regular_df):
    wins = regular_df[regular_df['Win'] == 1].copy()
    return (
        wins[['Season', 'DayNum', 'TeamID', 'OppTeamID', 'Margin']]
        .rename(columns={'TeamID': 'WTeamID', 'OppTeamID': 'LTeamID'})
        .drop_duplicates(subset=['Season', 'DayNum', 'WTeamID', 'LTeamID'])
        .reset_index(drop=True)
    )

print('Computando ELO ratings (masculino)...')
elo_m = precompute_starting_elo(
    _elo_games(games_m),
    k_factor=CONFIG['elo_k_factor'],
    initial_rating=CONFIG['elo_initial_rating'],
    carry_factor=CONFIG['elo_carry_factor'],
    use_margin=CONFIG.get('elo_use_margin', True),
    margin_cap=CONFIG.get('elo_margin_cap', 30.0),
)
print(f'  Entradas ELO M: {len(elo_m)}')

# Converte dict {(season, team_id): elo} → DataFrame para visualização
elo_df = pd.DataFrame(
    [{'Season': s, 'TeamID': t, 'ELO': v} for (s, t), v in elo_m.items()]
).sort_values(['TeamID', 'Season'])

print(f'\n📊 ELO DataFrame: {elo_df.shape}')
print(f'   Temporadas: {sorted(elo_df["Season"].unique())}')
print('\n✅ ELO computado!')
elo_df.head(10)

In [ ]:
# Distribuição de ELO por temporada
elo_stats = elo_df.groupby('Season')['ELO'].describe()[['mean', 'std', 'min', 'max']].round(1)
print('📊 ELO por Temporada:')
print(elo_stats)

fig, ax = plt.subplots(figsize=(13, 5))
seasons_sorted = sorted(elo_df['Season'].unique())
data_to_plot = [elo_df[elo_df['Season'] == s]['ELO'].values for s in seasons_sorted]
ax.boxplot(data_to_plot, labels=seasons_sorted)
ax.set_title('Distribuição de ELO Ratings por Temporada')
ax.set_xlabel('Season')
ax.set_ylabel('ELO Rating')
ax.axhline(CONFIG['elo_initial_rating'], color='red', linestyle='--', alpha=0.5, label=f'Initial = {CONFIG["elo_initial_rating"]}')
ax.legend()
plt.tight_layout()
plt.show()

## 5️⃣ Feature Engineering (um fold de exemplo)

In [ ]:
from src.features import build_team_features, attach_team_features, make_matchup_features
from src.data import prepare_eval_games

DEMO_SEASON = 2024
print(f'Construindo features para Season {DEMO_SEASON}...')

feats_m = build_team_features(
    games_m, DEMO_SEASON,
    CONFIG['recent_games_window'],
    CONFIG['alpha_ci'],
    starting_elo=elo_m,
    cfg=CONFIG,
)
print(f'  Features masculino: {feats_m.shape}')

eval_df = prepare_eval_games(tourney_m, DEMO_SEASON, 'M')

# Para demo, seeds feminino vazio (só masculino)
matchup_df = attach_team_features(eval_df, feats_m, feats_m, seeds_m, seeds_m, CONFIG)
matchup_df = make_matchup_features(matchup_df, cfg=CONFIG)

print(f'  Matchup frame: {matchup_df.shape}')
print(f'  Colunas: {list(matchup_df.columns[:10])} ...')
print('\n✅ Features OK!')
matchup_df[['TeamIDLow', 'TeamIDHigh', 'ActualLowWin', 'elo_diff', 'seed_diff', 'season_margin_diff']].head(10)

## 6️⃣ Métricas de Calibração (demo com dados sintéticos)

In [ ]:
np.random.seed(42)
n = 1000
y_true = np.random.binomial(1, 0.5, n)

# Simula predições correlacionadas mas sem calibrar
y_raw = np.clip((np.random.uniform(0, 1, n) + 2 * y_true) / 3, 0, 1)

# Calibra com Platt scaling
cal = LogisticRegression()
cal.fit(y_raw.reshape(-1, 1), y_true)
y_cal = cal.predict_proba(y_raw.reshape(-1, 1))[:, 1]

metrics = {
    'Brier (raw)':    brier_score_loss(y_true, y_raw),
    'Brier (cal)':    brier_score_loss(y_true, y_cal),
    'AUC (cal)':      roc_auc_score(y_true, y_cal),
    'LogLoss (cal)':  log_loss(y_true, y_cal),
}

print('📊 PERFORMANCE METRICS')
print('=' * 40)
for k, v in metrics.items():
    print(f'  {k:<20} {v:.4f}')
print(f'  {"Melhoria Brier":<20} {metrics["Brier (raw)"] - metrics["Brier (cal)"]:.4f}')
print('=' * 40)

## 7️⃣ Calibration Curves

In [ ]:
pt_raw, pp_raw = calibration_curve(y_true, y_raw, n_bins=10)
pt_cal, pp_cal = calibration_curve(y_true, y_cal, n_bins=10)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

for ax, pt, pp, label, color, brier in [
    (ax1, pt_raw, pp_raw, 'Sem Calibração', '#E6A23C', metrics['Brier (raw)']),
    (ax2, pt_cal, pp_cal, 'Platt Scaling',  '#43B89C', metrics['Brier (cal)']),
]:
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Perfeita')
    ax.plot(pp, pt, 'o-', linewidth=2, markersize=7, color=color, label=label)
    ax.fill_between(pp, pt, pp, alpha=0.15, color=color)
    ax.set_xlabel('Probabilidade Predita')
    ax.set_ylabel('Fração de Positivos')
    ax.set_title(f'{label}\nBrier = {brier:.4f}')
    ax.legend(fontsize=10)
    ax.grid(alpha=0.3)

plt.suptitle('Calibration Curves — antes e depois', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('✅ Calibração melhora o Brier Score significativamente!')

## 8️⃣ Resumo da Configuração Ativa

In [ ]:
import json

print('📋 PREDICTO CONFIG (valores ativos)')
print('=' * 55)
keys_to_show = [
    'target_season', 'backtest_seasons',
    'elo_k_factor', 'elo_carry_factor', 'elo_use_margin',
    'poisson_shrinkage', 'pred_clip_min', 'pred_clip_max',
    'manual_temperature',
]
for k in keys_to_show:
    v = CONFIG.get(k, '—')
    print(f'  {k:<28} {v}')

print('\nBlend weights:')
for k, v in CONFIG.get('blend_weights', {}).items():
    bar = '█' * int(v * 40)
    print(f'  {k:<10} {bar:<40} {v:.2f}')

print('=' * 55)
print(f'\n  data_dir: {DATA_DIR}')

## 9️⃣ Como Rodar o Pipeline Completo

In [ ]:
print('🚀 COMANDOS PARA EXECUTAR O PIPELINE:')
print()
print('  # Full pipeline (backtest + submission):')
print('  python scripts/run_pipeline_2026.py')
print()
print('  # Apenas backtest:')
print('  python scripts/run_pipeline_2026.py --mode backtest')
print()
print('  # Apenas submission:')
print('  python scripts/run_pipeline_2026.py --mode submit')
print()
print('  # Servidor de calibração (UI React):')
print('  uvicorn scripts.calibration_server:app --reload --port 8787')
print()
print('─' * 50)
print('  Docs: PROVENANCE.md | Audit: AUDITORIA_TECNICA_v3.md')

---

## Resumo

| Etapa | Status |
|---|---|
| Imports e path | ✅ |
| Detecção automática de DATA_DIR | ✅ |
| Carregamento de dados reais | ✅ |
| ELO ratings cross-seasonal | ✅ |
| Feature engineering (matchup frame) | ✅ |
| Métricas Brier / AUC / LogLoss | ✅ |
| Calibration curves | ✅ |
| Config overview | ✅ |

**Version:** 3.1 — **Author:** João Pedro Passos Tocantins — **License:** MIT